# Hopf Oscillator — x(t) vs y(t) Model Comparison

This notebook walks through the full multi-model training and evaluation pipeline
for the Hopf physical reservoir computer.

The Hopf oscillator produces two states — **x(t)** and **y(t)**. Published
papers (Shougat et al. 2021, 2023) use only x(t) and discard y(t), noting
it *"likely stores information."* This notebook explores what y(t) contributes.

## Models compared

| Model | Input | Purpose |
|-------|-------|---------|
| A — `cnn_x_only` | x(t) only | Baseline (paper 2) |
| B — `cnn_xy_dual` | x(t)+y(t) two channels | Does y add signal? |
| C — `cnn_phase` | sqrt(x²+y²) orbit radius | Amplitude only |
| D — `cnn_angle` | arctan2(y,x) phase angle | Rotation only |
| E — `cnn_xy_fusion` | late fusion x branch + y branch | Independent representations |
| F — `depthwise_cnn` | x+y depthwise separable | M33/M55 optimised |
| G — `reservoir_readout` | ridge regression | Paper 1 linear baseline |
| H — `ensemble` | majority vote over A/B/C | Best accuracy |


## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add repo root to path
REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10

print(f"Repo root: {REPO_ROOT}")
print(f"Python: {sys.version.split()[0]}")

## 2. Generate Hopf Oscillator Data (x and y)

Both x(t) and y(t) come from the **same** single integration — no extra cost.

In [ ]:
from data.sample_data import generate_dataset_xy, CLASS_NAMES

N_CLIPS = 20   # small set for notebook speed; use 100 for full training
N_CLASSES = 5

print(f"Generating {N_CLIPS} clips/class × {N_CLASSES} classes ...")
raw_x, raw_y, labels = generate_dataset_xy(
    n_clips_per_class=N_CLIPS, n_classes=N_CLASSES, cache=False,
)

print(f"\n  raw_x: {raw_x.shape}  ({raw_x.dtype})")
print(f"  raw_y: {raw_y.shape}  ({raw_y.dtype})")
print(f"  labels: {labels.shape}  classes: {np.unique(labels)}")

# Show one clip: x(t) and y(t) together form the spinning dot
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
t = np.linspace(0, 1, raw_x.shape[1])
axes[0].plot(t[:2000], raw_x[0, :2000], lw=0.5, label="x(t)")
axes[0].set_xlabel("Time (s)")
axes[0].set_title(f"Raw x(t) — class 0: {CLASS_NAMES[0]}")
axes[1].plot(t[:2000], raw_y[0, :2000], lw=0.5, color="coral", label="y(t)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title(f"Raw y(t) — class 0: {CLASS_NAMES[0]}")
plt.tight_layout()
plt.show()

# Phase portrait: the spinning dot
fig, ax = plt.subplots(figsize=(5, 5))
for cls in range(N_CLASSES):
    idx = np.where(labels == cls)[0][0]
    ax.plot(raw_x[idx, :5000], raw_y[idx, :5000], lw=0.4,
            alpha=0.7, label=CLASS_NAMES[cls])
ax.set_xlabel("x(t)")
ax.set_ylabel("y(t)")
ax.set_title("Phase portrait: Hopf limit cycle (first 50ms)")
ax.legend(fontsize=8)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 3. Ingest: Downsample → Normalise → atanh → Reshape

In [ ]:
from pipeline.ingest import process_dataset
from pipeline.features_xy import extract_y_features

print("Processing x(t) through ingest pipeline ...")
x_feat = process_dataset(raw_x)     # (n, 200, 100) float64
print(f"  x_feat: {x_feat.shape}  range=[{x_feat.min():.3f}, {x_feat.max():.3f}]")

print("Processing y(t) through identical ingest pipeline ...")
y_feat = extract_y_features(raw_y)  # (n, 200, 100) float64
print(f"  y_feat: {y_feat.shape}  range=[{y_feat.min():.3f}, {y_feat.max():.3f}]")

print("\ny(t) receives NO special treatment — same downsample/normalise/atanh pipeline.")

## 4. Compute All Input Representations

In [ ]:
from pipeline.features_xy import extract_all_representations

reps = extract_all_representations(x_feat, y_feat)

print("Input representations:")
for name, arr in reps.items():
    print(f"  {name:12s}: shape={arr.shape}  dtype={arr.dtype}  "
          f"range=[{arr.min()}, {arr.max()}]")

## 5. Visualise Feature Maps: x vs y vs phase vs angle

In [ ]:
rep_keys = ["x_only", "y_only", "phase", "angle"]
rep_labels = [
    "x(t) — published input",
    "y(t) — discarded state",
    "r(t) = √(x²+y²) orbit radius",
    "θ(t) = arctan2(y,x) phase angle",
]

fig, axes = plt.subplots(len(rep_keys), N_CLASSES,
                          figsize=(3 * N_CLASSES, 3 * len(rep_keys)))

cmaps = ["gray", "gray", "plasma", "twilight_shifted"]

for row, (rep_key, rep_label, cmap) in enumerate(zip(rep_keys, rep_labels, cmaps)):
    for cls in range(N_CLASSES):
        mask = labels == cls
        idx = np.where(mask)[0][0]
        ax = axes[row, cls]
        ax.imshow(reps[rep_key][idx], cmap=cmap, aspect="auto")
        if row == 0:
            ax.set_title(f"class {cls}\n{CLASS_NAMES[cls]}", fontsize=9)
        if cls == 0:
            ax.set_ylabel(rep_label, fontsize=8)
        ax.axis("off")

plt.suptitle("Feature Representations — One Clip per Class",
             fontsize=13, y=1.01, fontweight="bold")
plt.tight_layout()
plt.show()

print("Observation:")
print("  x(t) and y(t) look visually similar but encode complementary information.")
print("  Phase angle θ(t) reveals frequency structure not visible in x or y alone.")
print("  Orbit radius r(t) reveals amplitude perturbations (class-specific energy).")

## 6. Inspect Model Architectures

In [ ]:
from pipeline.models.cnn_x_only import build_model as build_a
from pipeline.models.cnn_xy_dual import build_model as build_b
from pipeline.models.cnn_xy_fusion import build_model as build_e
from pipeline.models.depthwise_cnn import build_model as build_f

model_a = build_a(n_classes=N_CLASSES)
model_b = build_b(n_classes=N_CLASSES)
model_e = build_e(n_classes=N_CLASSES)
model_f = build_f(n_classes=N_CLASSES)

print("=" * 50)
print("Model A — x only baseline")
model_a.summary()

print("\n" + "=" * 50)
print("Model B — two-channel x+y")
model_b.summary()

print("\n" + "=" * 50)
print("Model F — depthwise separable (M55 optimised)")
model_f.summary()

# Parameter count comparison
models_for_compare = {
    "A — x only": model_a,
    "B — xy dual": model_b,
    "F — depthwise": model_f,
}
print("\nParameter comparison:")
for name, m in models_for_compare.items():
    print(f"  {name:20s}: {m.count_params():>10,} params")

## 7. Quick Training (small dataset — for notebook demonstration)

For full training use `pipeline/train_all.py` which trains all 8 models with QAT.

In [ ]:
from pipeline.train_all import TrainConfig, _prepare_split, _apply_qat, _train_keras_model
import tensorflow as tf

cfg = TrainConfig(
    epochs=15,
    batch_size=8,
    n_classes=N_CLASSES,
    use_qat=False,   # skip QAT for notebook speed
    val_split=0.2,
)

ckpt_dir = REPO_ROOT / "pipeline" / "output" / "notebook_ckpts"
ckpt_dir.mkdir(parents=True, exist_ok=True)

histories = {}

# Train Model A — x only
print("Training Model A — x only ...")
x_tr_a, y_tr_a, x_vl_a, y_vl_a = _prepare_split(
    reps["x_only"], labels, cfg.val_split, cfg.n_classes
)
model_a_trained = build_a(n_classes=N_CLASSES)
h_a, _ = _train_keras_model(
    model_a_trained, x_tr_a, y_tr_a, x_vl_a, y_vl_a,
    cfg, ckpt_dir / "model_a_best.keras",
)
histories["A — x only"] = h_a

# Train Model B — xy dual
print("\nTraining Model B — xy dual ...")
x_tr_b, y_tr_b, x_vl_b, y_vl_b = _prepare_split(
    reps["xy_dual"], labels, cfg.val_split, cfg.n_classes, expand_channel=False
)
model_b_trained = build_b(n_classes=N_CLASSES)
h_b, _ = _train_keras_model(
    model_b_trained, x_tr_b, y_tr_b, x_vl_b, y_vl_b,
    cfg, ckpt_dir / "model_b_best.keras",
)
histories["B — xy dual"] = h_b

# Train Model C — phase
from pipeline.models.cnn_phase import build_model as build_c
print("\nTraining Model C — orbit radius ...")
x_tr_c, y_tr_c, x_vl_c, y_vl_c = _prepare_split(
    reps["phase"], labels, cfg.val_split, cfg.n_classes
)
model_c_trained = build_c(n_classes=N_CLASSES)
h_c, _ = _train_keras_model(
    model_c_trained, x_tr_c, y_tr_c, x_vl_c, y_vl_c,
    cfg, ckpt_dir / "model_c_best.keras",
)
histories["C — orbit radius"] = h_c

# Train Model D — angle
from pipeline.models.cnn_angle import build_model as build_d
print("\nTraining Model D — phase angle ...")
x_tr_d, y_tr_d, x_vl_d, y_vl_d = _prepare_split(
    reps["angle"], labels, cfg.val_split, cfg.n_classes
)
model_d_trained = build_d(n_classes=N_CLASSES)
h_d, _ = _train_keras_model(
    model_d_trained, x_tr_d, y_tr_d, x_vl_d, y_vl_d,
    cfg, ckpt_dir / "model_d_best.keras",
)
histories["D — phase angle"] = h_d

## 8. Ridge Regression Baseline (Model G)

In [ ]:
from pipeline.models.reservoir_readout import fit_all_readouts

n_total = len(labels)
n_val = int(n_total * 0.2)
idx = np.random.default_rng(0).permutation(n_total)
val_idx, train_idx = idx[:n_val], idx[n_val:]

ridge_results = fit_all_readouts(
    x_features=reps["x_only"][train_idx],
    y_features=reps["y_only"][train_idx],
    labels_train=labels[train_idx],
    x_val=reps["x_only"][val_idx],
    y_val=reps["y_only"][val_idx],
    labels_val=labels[val_idx],
)

print("\nRidge regression results:")
for variant, res in ridge_results.items():
    print(f"  G — ridge {variant:20s}: val_accuracy = {res['val_acc']:.4f}")

## 9. Compare Training Curves

In [ ]:
n_models = len(histories)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4), sharey=True)

colours = ["steelblue", "coral", "mediumseagreen", "mediumpurple"]

for ax, (model_name, hist), colour in zip(axes, histories.items(), colours):
    epochs = range(1, len(hist.history["accuracy"]) + 1)
    ax.plot(epochs, hist.history["accuracy"], "-", color=colour,
            label="train", lw=2)
    ax.plot(epochs, hist.history["val_accuracy"], "--", color=colour,
            label="val", lw=2, alpha=0.7)
    ax.set_title(f"Model {model_name}", fontsize=10)
    ax.set_xlabel("Epoch")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Accuracy")
plt.suptitle("Training Curves — Accuracy per Epoch", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 10. Accuracy Comparison Bar Chart

In [ ]:
# Collect best val accuracy per model
model_val_accs = {}
for model_name, hist in histories.items():
    model_val_accs[model_name] = max(hist.history["val_accuracy"])

# Add ridge results
for variant, res in ridge_results.items():
    model_val_accs[f"G — ridge\n{variant}"] = res["val_acc"]

names = list(model_val_accs.keys())
accs = list(model_val_accs.values())

colours_bar = [
    "steelblue", "coral", "mediumseagreen", "mediumpurple",
    "slategray", "slategray", "slategray",
]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(accs)), accs,
              color=colours_bar[:len(accs)], alpha=0.85, edgecolor="white",
              width=0.65)
ax.set_xticks(range(len(accs)))
ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel("Validation Accuracy", fontsize=11)
ax.set_title("Hopf Oscillator — Multi-Model Accuracy Comparison",
             fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.1)
ax.axhline(y=accs[0], color="steelblue", lw=1.5, linestyle="--",
           alpha=0.5, label="Model A baseline")
ax.legend(fontsize=9)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{acc:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Print key comparison findings
print("\nKey findings (short training, interpret with caution):")
a_acc = model_val_accs.get("A — x only", None)
b_acc = model_val_accs.get("B — xy dual", None)
if a_acc and b_acc:
    delta = b_acc - a_acc
    print(f"  A vs B (does y improve accuracy?):  Δ = {delta:+.4f}")
c_acc = model_val_accs.get("C — orbit radius", None)
if a_acc and c_acc:
    delta = c_acc - a_acc
    print(f"  A vs C (orbit radius vs x only):    Δ = {delta:+.4f}")
d_acc = model_val_accs.get("D — phase angle", None)
if a_acc and d_acc:
    delta = d_acc - a_acc
    print(f"  A vs D (phase angle vs x only):     Δ = {delta:+.4f}")

## 11. Ensemble — Majority Vote (Model H)

In [ ]:
from pipeline.models.ensemble import VoteEnsemble

# Use trained models A, B, C, D for ensemble
x_val_a = np.expand_dims(reps["x_only"][val_idx].astype(np.float32), -1)
x_val_b = reps["xy_dual"][val_idx].astype(np.float32)
x_val_c = np.expand_dims(reps["phase"][val_idx].astype(np.float32), -1)
x_val_d = np.expand_dims(reps["angle"][val_idx].astype(np.float32), -1)
labels_val = labels[val_idx]

ensemble_majority = VoteEnsemble(strategy="majority")
ensemble_majority.add_keras_model(
    model_a_trained, x_val_a, labels_val, input_key="x_only", name="Model A")
ensemble_majority.add_keras_model(
    model_b_trained, x_val_b, labels_val, input_key="xy_dual", name="Model B")
ensemble_majority.add_keras_model(
    model_c_trained, x_val_c, labels_val, input_key="phase", name="Model C")

ensemble_weighted = VoteEnsemble(strategy="weighted")
ensemble_weighted.add_keras_model(
    model_a_trained, x_val_a, labels_val, input_key="x_only", name="Model A")
ensemble_weighted.add_keras_model(
    model_b_trained, x_val_b, labels_val, input_key="xy_dual", name="Model B")
ensemble_weighted.add_keras_model(
    model_c_trained, x_val_c, labels_val, input_key="phase", name="Model C")

inputs_val = {
    "x_only": x_val_a,
    "xy_dual": x_val_b,
    "phase": x_val_c,
    "angle": x_val_d,
}

acc_majority = ensemble_majority.score(inputs_val, labels_val)
acc_weighted = ensemble_weighted.score(inputs_val, labels_val)

print(f"\nEnsemble results:")
print(f"  H — majority vote:   {acc_majority:.4f}")
print(f"  H — weighted vote:   {acc_weighted:.4f}")
if a_acc:
    print(f"  Model A baseline:    {a_acc:.4f}")
    print(f"  Ensemble gain (majority): {acc_majority - a_acc:+.4f}")

ensemble_majority.member_summary()

## 12. Full Training and Evaluation (run externally)

For a full 100-epoch training run with QAT and all 8 models, run:

```bash
# Full training (100 clips/class, 100 epochs, QAT enabled)
python -m pipeline.train_all

# Or with custom config:
python -m pipeline.train_all --epochs 100 --n_clips 100 --n_classes 5

# Evaluation report + plots:
python -m pipeline.evaluate --checkpoint_dir checkpoints/ --output_dir output/eval/

# TFLite INT8 conversion + firmware header generation:
python -m pipeline.convert_all --checkpoint_dir checkpoints/
```

Generated artefacts in `firmware/`:
- `model_a_cnn_x_only.tflite` — TFLite INT8 flatbuffer
- `cmsis_nn_params_model_a_cnn_x_only.h` — weights + per-channel quant params
- `model_data_model_a_cnn_x_only.cc` — C byte array for TFLite Micro
- *(one set per model)*

---

## Summary of Representations

| Representation | Physical meaning | What CNN learns |
|----------------|------------------|-----------------|
| x(t) only | Projection of oscillator onto x-axis | Temporal patterns in x dynamics |
| y(t) only | Quadrature state (90° offset) | Complementary temporal patterns |
| r(t) orbit radius | Distance of dot from centre | Amplitude/energy of perturbation |
| θ(t) phase angle | Angular position of dot | Frequency structure, timing |
| x+y dual channel | Both states simultaneously | Cross-channel x/y relationships |
| Late fusion x‖y | Independent x and y branches | Specialised per-modality features |
